In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv("Dataset.csv")
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [3]:
# Basic Cleaning
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

df.drop_duplicates(inplace=True)

In [4]:
# Missing value treatment
df['review_rating'] = df['review_rating'].fillna(df['review_rating'].median())

In [5]:
# Encoding of Yes/No columns

binary_cols = ["subscription_status","discount_applied","promo_code_used"]

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [6]:
# Create age group
df['age_group'] = pd.cut(
    df['age'],
    bins=[0,25,35,45,55,100],
    labels=['18-25','26-35','36-45','46-55','55+']
)

In [7]:
# Frequency Score
freq_map = {
    'Weekly':10,
    'Fortnightly':8,
    'Monthly':6,
    'Quarterly':4,
    'Annually':2
}

df['frequency_score'] =df['frequency_of_purchases'].map(freq_map)


In [8]:
# Satisfaction Flag
df['satisfaction_flag'] = np.where(df['review_rating'] >= 4,1,0)

# Rating Segment
def rating_segment(x):
    if x >= 4.5:
        return "Highly Satisfied"
    elif x >= 3:
        return "Satisfied"
    else:
        return "Unsatisfied"

df['rating_segment'] = df['review_rating'].apply(rating_segment)

In [9]:
# Dependency score
df['dependency_score'] =(df['discount_applied'] + df['promo_code_used'])/2

# Dependency Segment 
def dependency_segment(x):
    if x >= 0.75:
        return "Highly Promo Dependent"
    elif x >= 0.5:
        return "Moderately Dependent"
    else:
        return "Loyal Customer"

df['dependency_segment'] = df['dependency_score'].apply(dependency_segment)

In [10]:
# Normalizing Important features
scaler = MinMaxScaler()

df[['purchase_norm']] = scaler.fit_transform(df[['purchase_amount_(usd)']])

df[['previous_purchase_norm']] = scaler.fit_transform(df[['previous_purchases']])

df[['review_norm']] = scaler.fit_transform(df[['review_rating']])

df[['frequency_norm']] = scaler.fit_transform(df[['frequency_score']])

In [11]:
# Customer value score and their value tier
df['customer_value_score'] = (df['purchase_norm'] + df['previous_purchase_norm'])/2

Q1 = df['customer_value_score'].quantile(.25)
Q2 = df['customer_value_score'].quantile(.50)
Q3 = df['customer_value_score'].quantile(.75)

def value_tier(x):
    if x >= Q3:
        return "High Value"
    elif x >= Q2:
        return "Medium Value"
    elif x >= Q1:
        return "Low Value"
    else:
        return "Very Low Value"

df['value_tier'] =df['customer_value_score'].apply(value_tier)

In [12]:
# Behavioral Loyalty
df['behavioral_loyalty_score'] = 0.40 * df['previous_purchase_norm'] + 0.40 * df['frequency_norm'] + 0.20 * df['subscription_status'] 

# Relationship Loyalty
df['relationship_loyalty_score'] = 0.35 * df['previous_purchase_norm'] + 0.25 * df['subscription_status'] + 0.25 * df['review_norm'] + 0.15 * df['purchase_norm']


In [13]:
# Now compare loyalty method
corr_1 = df[['behavioral_loyalty_score','customer_value_score']].corr().iloc[0,1]

corr_2 = df[['relationship_loyalty_score','customer_value_score']].corr().iloc[0,1]

print("Behavioral Loyalty Correlation :", round(corr_1,3))
print("Relationship Loyalty Correlation :", round(corr_2,3))

Behavioral Loyalty Correlation : 0.394
Relationship Loyalty Correlation : 0.615


In [14]:
# Select Best Loyalty Model
if corr_1 > corr_2:

    df['loyalty_score'] = df['behavioral_loyalty_score']
    selected_model = "Behavioral Loyalty"

else:
    df['loyalty_score'] = df['relationship_loyalty_score']
    selected_model = "Relationship Loyalty"

print("Selected Model :", selected_model)

Selected Model : Relationship Loyalty


In [15]:
# Loyalty Segment
q1 = df['loyalty_score'].quantile(.25)
q2 = df['loyalty_score'].quantile(.50)
q3 = df['loyalty_score'].quantile(.75)

def loyalty_segment(x):
    if x >= q3:
        return "Champion"
    elif x >= q2:
        return "Loyal"
    elif x >= q1:
        return "Potential Loyalist"
    else:
        return "At Risk"

df['loyalty_segment'] =df['loyalty_score'].apply(loyalty_segment)

In [16]:
# Promo User Flag
df['high_promo_user'] = np.where(df['dependency_score'] >= 0.75,1,0)

In [17]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_(usd),location,size,color,season,...,previous_purchase_norm,review_norm,frequency_norm,customer_value_score,value_tier,behavioral_loyalty_score,relationship_loyalty_score,loyalty_score,loyalty_segment,high_promo_user
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,0.265306,0.24,0.75,0.338903,Very Low Value,0.606122,0.464732,0.464732,Loyal,1
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,0.020408,0.24,0.75,0.285204,Very Low Value,0.508163,0.399643,0.399643,Potential Loyalist,1
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,0.448980,0.24,1.00,0.555740,Medium Value,0.779592,0.566518,0.566518,Champion,1
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,0.979592,0.40,1.00,0.927296,High Value,0.991837,0.824107,0.824107,Champion,1
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,0.612245,0.08,0.00,0.487372,Low Value,0.444898,0.538661,0.538661,Loyal,1


In [18]:
# Final columns for further Deliverables
final_df = df[[
    'customer_id',
    'age',
    'age_group',
    'gender',
    'category',
    'purchase_amount_(usd)',
    'location',
    'season',
    'review_rating',
    'subscription_status',
    'discount_applied',
    'promo_code_used',
    'previous_purchases',
    'payment_method',
    'frequency_of_purchases',
    'frequency_score',
    'dependency_score',
    'dependency_segment',
    'customer_value_score',
    'value_tier',
    'behavioral_loyalty_score',
    'relationship_loyalty_score',
    'loyalty_score',
    'loyalty_segment',
    'satisfaction_flag',
    'rating_segment',
    'high_promo_user'
]]

In [19]:
final_df.head()

,customer_id,age,age_group,gender,category,purchase_amount_(usd),location,season,review_rating,subscription_status,...,dependency_segment,customer_value_score,value_tier,behavioral_loyalty_score,relationship_loyalty_score,loyalty_score,loyalty_segment,satisfaction_flag,rating_segment,high_promo_user
0,1,55,46-55,Male,Clothing,53,Kentucky,Winter,3.1,1,...,Highly Promo Dependent,0.338903,Very Low Value,0.606122,0.464732,0.464732,Loyal,0,Satisfied,1
1,2,19,18-25,Male,Clothing,64,Maine,Winter,3.1,1,...,Highly Promo Dependent,0.285204,Very Low Value,0.508163,0.399643,0.399643,Potential Loyalist,0,Satisfied,1
2,3,50,46-55,Male,Clothing,73,Massachusetts,Spring,3.1,1,...,Highly Promo Dependent,0.555740,Medium Value,0.779592,0.566518,0.566518,Champion,0,Satisfied,1
3,4,21,18-25,Male,Footwear,90,Rhode Island,Spring,3.5,1,...,Highly Promo Dependent,0.927296,High Value,0.991837,0.824107,0.824107,Champion,0,Satisfied,1
4,5,45,36-45,Male,Clothing,49,Oregon,Spring,2.7,1,...,Highly Promo Dependent,0.487372,Low Value,0.444898,0.538661,0.538661,Loyal,0,Unsatisfied,1


In [20]:
# New datset for SQL

final_df.to_csv("customer_intelligence_dataset.csv",index=False)


print("Dataset Saved Successfully")
print(final_df.shape)

Dataset Saved Successfully
(3900, 27)
